In [ ]:
%%js

// GAME_RUNNER: Tower of Treasure - move with WASD, walk to the puzzle station, press E to solve syntax, unlock the door, and climb the tower.

import GameControl from '/assets/js/GameEnginev1/essentials/GameControl.js';
import GameEnvBackground from '/assets/js/GameEnginev1/essentials/GameEnvBackground.js';
import Player from '/assets/js/GameEnginev1/essentials/Player.js';

/* -----------------------------
   Cleanup if re-run
------------------------------ */
if (window.__towerCleanup) {
  try {
    window.__towerCleanup();
  } catch (e) {
    console.warn("Cleanup error:", e);
  }
  window.__towerCleanup = null;
}

/* -----------------------------
   Puzzle Data
------------------------------ */
const PUZZLES = [
  {
    id: 1,
    title: "Floor 1: Variable Declaration",
    prompt: "Create an int variable named score with value 10.",
    blocks: ["int", "score", "=", "10", ";"],
    solution: ["int", "score", "=", "10", ";"],
    explanation: "A Java variable declaration needs a type, variable name, value, and semicolon."
  },
  {
    id: 2,
    title: "Floor 2: Print Statement",
    prompt: "Print the variable score.",
    blocks: ["System.out.println", "(", "score", ")", ";"],
    solution: ["System.out.println", "(", "score", ")", ";"],
    explanation: "Java print statements use System.out.println(expression);"
  },
  {
    id: 3,
    title: "Floor 3: If Statement",
    prompt: "Write the opening of an if statement that checks whether score is greater than 5.",
    blocks: ["if", "(", "score", ">", "5", ")", "{"],
    solution: ["if", "(", "score", ">", "5", ")", "{"],
    explanation: "if statements use if (condition) {"
  }
];

/* -----------------------------
   Helpers
------------------------------ */
function makeEl(tag, styles, text) {
  const el = document.createElement(tag);
  Object.assign(el.style, styles || {});
  if (text) el.textContent = text;
  return el;
}

function shuffleArray(arr) {
  const copy = arr.slice();
  for (let i = copy.length - 1; i > 0; i--) {
    const j = Math.floor(Math.random() * (i + 1));
    const temp = copy[i];
    copy[i] = copy[j];
    copy[j] = temp;
  }
  return copy;
}

function tokensEqual(a, b) {
  return JSON.stringify(a) === JSON.stringify(b);
}

function rectsOverlap(a, b) {
  return (
    a.x < b.x + b.width &&
    a.x + a.width > b.x &&
    a.y < b.y + b.height &&
    a.y + a.height > b.y
  );
}

function centerText(ctx, text, x, y) {
  ctx.textAlign = "center";
  ctx.textBaseline = "middle";
  ctx.fillText(text, x, y);
}

function getPlayer(gameEnv) {
  if (!gameEnv.gameObjects) return null;
  return gameEnv.gameObjects.find(function(obj) {
    return obj instanceof Player || obj.id === "Hero";
  });
}

function fadeToBlackThen(action) {
  const oldFade = document.getElementById("tower-fade-overlay");
  if (oldFade) oldFade.remove();

  const overlay = document.createElement("div");
  overlay.id = "tower-fade-overlay";
  overlay.style.position = "fixed";
  overlay.style.inset = "0";
  overlay.style.background = "black";
  overlay.style.opacity = "0";
  overlay.style.pointerEvents = "none";
  overlay.style.zIndex = "99999";
  overlay.style.transition = "opacity 0.45s ease";

  document.body.appendChild(overlay);

  requestAnimationFrame(() => {
    overlay.style.opacity = "1";
  });

  setTimeout(() => {
    action();

    setTimeout(() => {
      const current = document.getElementById("tower-fade-overlay");
      if (current) current.remove();
    }, 500);
  }, 450);
}



/* -----------------------------
   DOM UI: Puzzle Popup
------------------------------ */
class SyntaxPuzzleUI {
  constructor(puzzle, onSuccess, onClose) {
    this.puzzle = puzzle;
    this.onSuccess = onSuccess;
    this.onClose = onClose;
    this.root = null;
    this.answerArea = null;
    this.bankArea = null;
    this.feedback = null;
    this.explanation = null;
    this.isOpen = false;
  }

  open() {
    if (this.isOpen) return;
    this.isOpen = true;

    this.injectStyles();

    const old = document.getElementById("tower-puzzle-overlay");
    if (old) old.remove();

    this.root = makeEl("div", {
      position: "fixed",
      inset: "0",
      background: "rgba(0,0,0,0.45)",
      display: "flex",
      alignItems: "center",
      justifyContent: "center",
      zIndex: "10000",
      fontFamily: "Arial, sans-serif"
    });
    this.root.id = "tower-puzzle-overlay";

    const card = makeEl("div", {
      width: "min(900px, 92vw)",
      background: "#614b24",
      borderRadius: "24px",
      border: "4px solid #6a4b2f",
      padding: "24px",
      boxShadow: "0 18px 50px rgba(0,0,0,0.25)"
    });

    const title = makeEl("h2", {
      margin: "0 0 10px 0",
      color: "#4b2f1b",
      fontSize: "30px"
    }, this.puzzle.title);

    const prompt = makeEl("p", {
      margin: "0 0 16px 0",
      color: "#d4c2b4",
      fontSize: "18px",
      lineHeight: "1.5",
      fontWeight: "700"
    }, this.puzzle.prompt);

    const answerLabel = makeEl("div", {
      fontWeight: "800",
      color: "#ab9382",
      marginBottom: "8px"
    }, "Build the code here");

    this.answerArea = makeEl("div");
    this.answerArea.className = "tower-answer-area";

    const bankLabel = makeEl("div", {
      fontWeight: "800",
      color: "#c7b7ab",
      margin: "14px 0 8px 0"
    }, "Block Bank");

    this.bankArea = makeEl("div");
    this.bankArea.className = "tower-bank-area";

    this.feedback = makeEl("div", {
      minHeight: "28px",
      marginTop: "14px",
      fontSize: "18px",
      fontWeight: "800",
      color: "#c2ad9d"
    });

    this.explanation = makeEl("div", {
      minHeight: "22px",
      marginTop: "6px",
      color: "#dbd1c8",
      fontSize: "14px"
    });

    const btnRow = makeEl("div", {
      display: "flex",
      gap: "10px",
      flexWrap: "wrap",
      marginTop: "16px"
    });

    const checkBtn = makeEl("button", {}, "Check Answer");
    checkBtn.className = "tower-btn tower-btn-green";

    const resetBtn = makeEl("button", {}, "Reset");
    resetBtn.className = "tower-btn tower-btn-brown";

    const closeBtn = makeEl("button", {}, "Close");
    closeBtn.className = "tower-btn tower-btn-red";

    btnRow.appendChild(checkBtn);
    btnRow.appendChild(resetBtn);
    btnRow.appendChild(closeBtn);

    card.appendChild(title);
    card.appendChild(prompt);
    card.appendChild(answerLabel);
    card.appendChild(this.answerArea);
    card.appendChild(bankLabel);
    card.appendChild(this.bankArea);
    card.appendChild(this.feedback);
    card.appendChild(this.explanation);
    card.appendChild(btnRow);

    this.root.appendChild(card);
    document.body.appendChild(this.root);

    this.renderBlocks();

    checkBtn.addEventListener("click", () => this.checkAnswer());
    resetBtn.addEventListener("click", () => this.renderBlocks());
    closeBtn.addEventListener("click", () => this.close(false));
  }

  injectStyles() {
    if (document.getElementById("tower-puzzle-styles")) return;

    const style = document.createElement("style");
    style.id = "tower-puzzle-styles";
    style.textContent =
      ".tower-answer-area, .tower-bank-area {" +
      "min-height: 82px;" +
      "border-radius: 16px;" +
      "padding: 14px;" +
      "display: flex;" +
      "gap: 10px;" +
      "flex-wrap: wrap;" +
      "align-items: flex-start;" +
      "}" +
      ".tower-answer-area {" +
      "border: 2px dashed #7a5c44;" +
      "background: rgba(250,248,240,1);" +
      "}" +
      ".tower-bank-area {" +
      "border: 2px solid #d4c6b8;" +
      "background: rgba(255,255,255,1);" +
      "}" +
      ".tower-token {" +
      "padding: 10px 14px;" +
      "border-radius: 12px;" +
      "border: 2px solid #5b4128;" +
      "background: linear-gradient(180deg, #ffe5a8 0%, #f6c66c 100%);" +
      "color: #2e2115;" +
      "font-weight: 800;" +
      "font-size: 15px;" +
      "cursor: pointer;" +
      "user-select: none;" +
      "box-shadow: 0 3px 0 #b37a2a;" +
      "}" +
      ".tower-btn {" +
      "border: none;" +
      "border-radius: 12px;" +
      "padding: 10px 16px;" +
      "font-size: 15px;" +
      "font-weight: 800;" +
      "cursor: pointer;" +
      "color: white;" +
      "}" +
      ".tower-btn-green { background: #2d8f53; }" +
      ".tower-btn-brown { background: #8b5e3c; }" +
      ".tower-btn-red { background: #b0413e; }";
    document.head.appendChild(style);
  }

  renderBlocks() {
    if (!this.answerArea || !this.bankArea) return;

    this.answerArea.innerHTML = "";
    this.bankArea.innerHTML = "";
    this.feedback.textContent = "";
    this.explanation.textContent = "";

    const shuffled = shuffleArray(this.puzzle.blocks);
    shuffled.forEach((token) => {
      this.bankArea.appendChild(this.createToken(token));
    });
  }

  createToken(token) {
    const el = makeEl("div", {}, token);
    el.className = "tower-token";
    el.dataset.value = token;

    el.addEventListener("click", () => {
      if (el.parentElement === this.bankArea) {
        this.answerArea.appendChild(el);
      } else {
        this.bankArea.appendChild(el);
      }
    });

    return el;
  }

  getAnswerTokens() {
    return Array.from(this.answerArea.children).map((el) => el.dataset.value);
  }

  checkAnswer() {
    const placed = this.getAnswerTokens();

    if (placed.length !== this.puzzle.solution.length) {
      this.feedback.style.color = "#9a4f00";
      this.feedback.textContent = "Use all the needed blocks first.";
      this.explanation.textContent = "Move every required token into the answer area.";
      return;
    }

    if (tokensEqual(placed, this.puzzle.solution)) {
      this.feedback.style.color = "#1f7a40";
      this.feedback.textContent = "Correct! The door is unlocked.";
      this.explanation.textContent = this.puzzle.explanation;

      setTimeout(() => {
        this.close(true);
      }, 700);
    } else {
      this.feedback.style.color = "#9d2222";
      this.feedback.textContent = "Not quite. Try again.";
      this.explanation.textContent = "Check the order carefully: punctuation, parentheses, and semicolons matter.";
    }
  }

  close(success) {
    if (this.root) this.root.remove();
    this.isOpen = false;
    if (success) {
      this.onSuccess && this.onSuccess();
    } else {
      this.onClose && this.onClose();
    }
  }

  destroy() {
    if (this.root) this.root.remove();
  }
}

/* -----------------------------
   HUD
------------------------------ */
class TowerHUD {
  constructor(data, gameEnv) {
    this.gameEnv = gameEnv;
    this.message = data.message || "Walk to the gold station and press E";
    this.floorName = data.floorName || "Floor";
  }

  setMessage(msg) {
    this.message = msg;
  }

  update() {
    const ctx = this.gameEnv.ctx;
    ctx.save();

    ctx.fillStyle = "rgba(20, 20, 30, 0.72)";
    ctx.fillRect(14, 14, 470, 80);

    ctx.strokeStyle = "#f4d27a";
    ctx.lineWidth = 3;
    ctx.strokeRect(14, 14, 470, 80);

    ctx.fillStyle = "white";
    ctx.font = "bold 22px Arial";
    ctx.fillText("Tower of Treasure - " + this.floorName, 28, 42);

    ctx.fillStyle = "#f6e7b0";
    ctx.font = "16px Arial";
    ctx.fillText(this.message, 28, 70);

    ctx.restore();
  }

  resize() {}

  destroy() {}
}

/* -----------------------------
   Door
------------------------------ */
class TowerDoor {
  constructor(data, gameEnv) {
    this.gameEnv = gameEnv;
    this.x = data.x;
    this.y = data.y;
    this.width = data.width || 80;
    this.height = data.height || 170;
    this.locked = data.locked ?? true;
    this.label = data.label || "Door";
    this.nextLevelIndex = data.nextLevelIndex;
    this.cooldown = false;
  }

  unlock() {
    this.locked = false;
  }

  getHUD() {
    if (!this.gameEnv.gameObjects) return null;
    return this.gameEnv.gameObjects.find((obj) => obj instanceof TowerHUD);
  }

  update() {
    const ctx = this.gameEnv.ctx;
    const player = getPlayer(this.gameEnv);

    ctx.save();

    ctx.fillStyle = this.locked ? "#7a4a26" : "#5dade2";
    ctx.fillRect(this.x, this.y, this.width, this.height);

    ctx.strokeStyle = "#2d1b10";
    ctx.lineWidth = 4;
    ctx.strokeRect(this.x, this.y, this.width, this.height);

    ctx.fillStyle = this.locked ? "#d4a32b" : "#ecf0f1";
    ctx.beginPath();
    ctx.arc(this.x + this.width - 16, this.y + this.height / 2, 5, 0, Math.PI * 2);
    ctx.fill();

    ctx.fillStyle = "white";
    ctx.font = "bold 15px Arial";
    centerText(
      ctx,
      this.locked ? "LOCKED" : (this.nextLevelIndex !== undefined ? "NEXT FLOOR" : "OPEN"),
      this.x + this.width / 2,
      this.y - 16
    );

    ctx.restore();

    if (!player) return;

    const playerBox = {
      x: player.position?.x ?? player.x ?? 0,
      y: player.position?.y ?? player.y ?? 0,
      width: player.width || 55,
      height: player.height || 70
    };

    const doorBox = {
      x: this.x,
      y: this.y,
      width: this.width,
      height: this.height
    };

    if (this.locked) {
      if (rectsOverlap(playerBox, doorBox)) {
        if ((player.position?.x ?? player.x ?? 0) < this.x) {
          if (player.position) player.position.x = this.x - playerBox.width - 2;
          else player.x = this.x - playerBox.width - 2;
        } else {
          if (player.position) player.position.x = this.x + this.width + 2;
          else player.x = this.x + this.width + 2;
        }

        if (player.velocity) {
          player.velocity.x = 0;
          player.velocity.y = 0;
        }
      }
    } else {
      if (rectsOverlap(playerBox, doorBox) && !this.cooldown) {
        this.cooldown = true;

        const hud = this.getHUD();
        if (hud) {
          if (this.nextLevelIndex !== undefined) {
            hud.setMessage("Climbing to the next floor...");
          } else {
            hud.setMessage("The final door is open!");
          }
        }

        if (this.nextLevelIndex !== undefined && this.gameEnv.gameControl) {
          fadeToBlackThen(() => {
            this.gameEnv.gameControl.currentLevelIndex = this.nextLevelIndex;
            if (this.gameEnv.gameControl.transitionToLevel) {
              this.gameEnv.gameControl.transitionToLevel();
            }
          });
        }
      }
    }
  }

  resize() {}

  destroy() {}
}

/* -----------------------------
   Puzzle Station
------------------------------ */
class PuzzleStation {
  constructor(data, gameEnv) {
    this.gameEnv = gameEnv;
    this.x = data.x;
    this.y = data.y;
    this.width = data.width || 72;
    this.height = data.height || 72;
    this.puzzle = data.puzzle;
    this.doorId = data.doorId;
    this.solved = false;
    this.playerNear = false;
    this.isPopupOpen = false;
    this.keyDownBound = this.handleKeyDown.bind(this);

    addEventListener("keydown", this.keyDownBound);
  }

  removeInteractKeyListeners() {
    removeEventListener("keydown", this.keyDownBound);
  }

  getDoor() {
    if (!this.gameEnv.gameObjects) return null;
    return this.gameEnv.gameObjects.find((obj) => {
      return obj instanceof TowerDoor && obj.label === this.doorId;
    });
  }

  getHUD() {
    if (!this.gameEnv.gameObjects) return null;
    return this.gameEnv.gameObjects.find((obj) => obj instanceof TowerHUD);
  }

  openPuzzle() {
    if (this.solved || this.isPopupOpen) return;

    this.isPopupOpen = true;
    const hud = this.getHUD();
    if (hud) hud.setMessage("Solve the syntax puzzle");

    const ui = new SyntaxPuzzleUI(
      this.puzzle,
      () => {
        this.solved = true;
        this.isPopupOpen = false;
        const door = this.getDoor();
        if (door) door.unlock();
        if (hud) hud.setMessage("Door unlocked - head through the stairs!");
      },
      () => {
        this.isPopupOpen = false;
        if (hud) hud.setMessage("Walk to the gold station and press E");
      }
    );

    this.ui = ui;
    ui.open();
  }

  handleKeyDown(e) {
    if (this.playerNear && !this.solved && !this.isPopupOpen && (e.key === "e" || e.key === "E")) {
      this.openPuzzle();
    }
  }

  update() {
    const ctx = this.gameEnv.ctx;
    const player = getPlayer(this.gameEnv);
    if (!player) return;

    const px = player.position?.x ?? player.x ?? 0;
    const py = player.position?.y ?? player.y ?? 0;
    const pw = player.width || 55;
    const ph = player.height || 70;

    const stationBox = {
      x: this.x,
      y: this.y,
      width: this.width,
      height: this.height
    };

    const playerBox = { x: px, y: py, width: pw, height: ph };

    this.playerNear = rectsOverlap(
      {
        x: playerBox.x - 20,
        y: playerBox.y - 20,
        width: playerBox.width + 40,
        height: playerBox.height + 40
      },
      stationBox
    );

    const hud = this.getHUD();
    if (!this.isPopupOpen) {
      if (this.playerNear && !this.solved) {
        hud && hud.setMessage("Press E to solve the puzzle");
      } else if (!this.solved) {
        hud && hud.setMessage("Walk to the gold station and press E");
      }
    }

    ctx.save();

    ctx.fillStyle = this.solved ? "#42b96f" : "#f2c94c";
    ctx.fillRect(this.x, this.y, this.width, this.height);

    ctx.strokeStyle = "#533723";
    ctx.lineWidth = 4;
    ctx.strokeRect(this.x, this.y, this.width, this.height);

    ctx.fillStyle = "#2c1d12";
    ctx.font = "bold 14px Arial";
    centerText(ctx, this.solved ? "DONE" : "PUZZLE", this.x + this.width / 2, this.y + this.height / 2);

    if (this.playerNear && !this.solved) {
      ctx.fillStyle = "rgba(20,20,30,0.82)";
      ctx.fillRect(this.x - 6, this.y - 40, 116, 28);
      ctx.fillStyle = "white";
      ctx.font = "bold 15px Arial";
      centerText(ctx, "Press E", this.x + this.width / 2 + 2, this.y - 26);
    }

    ctx.restore();
  }

  resize() {}

  destroy() {
    this.removeInteractKeyListeners();
    if (this.ui) this.ui.destroy();
  }
}

/* -----------------------------
   Treasure Zone
------------------------------ */
class TreasureZone {
  constructor(data, gameEnv) {
    this.gameEnv = gameEnv;
    this.x = data.x;
    this.y = data.y;
    this.width = data.width || 120;
    this.height = data.height || 120;
    this.won = false;
    this.doorId = data.doorId || null;
  }

  getHUD() {
    if (!this.gameEnv.gameObjects) return null;
    return this.gameEnv.gameObjects.find((obj) => obj instanceof TowerHUD);
  }

  getDoor() {
    if (!this.doorId || !this.gameEnv.gameObjects) return null;
    return this.gameEnv.gameObjects.find((obj) => {
      return obj instanceof TowerDoor && obj.label === this.doorId;
    });
  }

  update() {
    const ctx = this.gameEnv.ctx;
    const player = getPlayer(this.gameEnv);
    const door = this.getDoor();
    const active = !door || !door.locked;

    ctx.save();
    ctx.fillStyle = active ? "#ffd95e" : "#9aa0a6";
    ctx.fillRect(this.x, this.y, this.width, this.height);
    ctx.strokeStyle = "#8c6310";
    ctx.lineWidth = 4;
    ctx.strokeRect(this.x, this.y, this.width, this.height);

    ctx.fillStyle = "#5a3d00";
    ctx.font = "bold 20px Arial";
    centerText(ctx, "Gem", this.x + this.width / 2, this.y + this.height / 2 - 14);
    ctx.font = "bold 14px Arial";
    centerText(ctx, active ? "TREASURE" : "LOCKED", this.x + this.width / 2, this.y + this.height / 2 + 18);
    ctx.restore();

    if (!player || this.won || !active) return;

    const playerBox = {
      x: player.position?.x ?? player.x ?? 0,
      y: player.position?.y ?? player.y ?? 0,
      width: player.width || 55,
      height: player.height || 70
    };

    const box = { x: this.x, y: this.y, width: this.width, height: this.height };

    if (rectsOverlap(playerBox, box)) {
      this.won = true;
      const hud = this.getHUD();
      if (hud) hud.setMessage("You found the treasure! Press Esc to end.");
      setTimeout(() => {
        alert("Treasure unlocked! You completed Tower of Treasure.");
      }, 100);
    }
  }

  resize() {}

  destroy() {}
}

/* -----------------------------
   Shared Player Data Helper
------------------------------ */
function makePlayerData(path) {
  return {
    id: "Hero",
    src: path + "/images/gamify/chillguy.png",
    SCALE_FACTOR: 5,
    STEP_FACTOR: 1000,
    ANIMATION_RATE: 50,
    INIT_POSITION: { x: 100, y: 300 },
    pixels: { height: 512, width: 384 },
    orientation: { rows: 4, columns: 3 },
    down: { row: 0, start: 0, columns: 3 },
    downRight: { row: 1, start: 0, columns: 3, rotate: Math.PI / 16 },
    downLeft: { row: 2, start: 0, columns: 3, rotate: -Math.PI / 16 },
    right: { row: 1, start: 0, columns: 3 },
    left: { row: 2, start: 0, columns: 3 },
    up: { row: 3, start: 0, columns: 3 },
    upRight: { row: 1, start: 0, columns: 3, rotate: -Math.PI / 16 },
    upLeft: { row: 2, start: 0, columns: 3, rotate: Math.PI / 16 },
    hitbox: { widthPercentage: 0.45, heightPercentage: 0.2 },
    keypress: { up: 87, left: 65, down: 83, right: 68 },
    label: "Explorer"
  };
}

/* -----------------------------
   Level 1
------------------------------ */
class Floor1Level {
  constructor(gameEnv) {
    const path = gameEnv.path;

    const bgData = {
      name: "custom_bg_1",
      src: path + "/images/gamebuilder/bg/tower.png",
      pixels: { height: 720, width: 1280 }
    };

    const hudData = {
      floorName: "Floor 1",
      message: "Walk to the gold station and press E"
    };

    const stationData = {
      x: 320,
      y: 300,
      width: 84,
      height: 84,
      puzzle: PUZZLES[0],
      doorId: "door1"
    };

    const doorData = {
      x: 610,
      y: 220,
      width: 90,
      height: 220,
      locked: true,
      label: "door1",
      nextLevelIndex: 1
    };

    this.classes = [
      { class: GameEnvBackground, data: bgData },
      { class: TowerHUD, data: hudData },
      { class: Player, data: makePlayerData(path) },
      { class: PuzzleStation, data: stationData },
      { class: TowerDoor, data: doorData }
    ];
  }
}

/* -----------------------------
   Level 2
------------------------------ */
class Floor2Level {
  constructor(gameEnv) {
    const path = gameEnv.path;

    const bgData = {
      name: "custom_bg_2",
      src: path + "/images/gamebuilder/bg/tower.png",
      pixels: { height: 720, width: 1280 }
    };

    const hudData = {
      floorName: "Floor 2",
      message: "Walk to the gold station and press E"
    };

    const stationData = {
      x: 320,
      y: 300,
      width: 84,
      height: 84,
      puzzle: PUZZLES[1],
      doorId: "door2"
    };

    const doorData = {
      x: 610,
      y: 220,
      width: 90,
      height: 220,
      locked: true,
      label: "door2",
      nextLevelIndex: 2
    };

    this.classes = [
      { class: GameEnvBackground, data: bgData },
      { class: TowerHUD, data: hudData },
      { class: Player, data: makePlayerData(path) },
      { class: PuzzleStation, data: stationData },
      { class: TowerDoor, data: doorData }
    ];
  }
}

/* -----------------------------
   Level 3
------------------------------ */
class Floor3Level {
  constructor(gameEnv) {
    const path = gameEnv.path;

    const bgData = {
      name: "custom_bg_3",
      src: path + "/images/gamebuilder/bg/tower.png",
      pixels: { height: 720, width: 1280 }
    };

    const hudData = {
      floorName: "Floor 3",
      message: "Solve the final puzzle and claim the treasure"
    };

    const stationData = {
      x: 320,
      y: 300,
      width: 84,
      height: 84,
      puzzle: PUZZLES[2],
      doorId: "door3"
    };

    const doorData = {
      x: 610,
      y: 220,
      width: 90,
      height: 220,
      locked: true,
      label: "door3"
    };

    const treasureData = {
      x: 1020,
      y: 280,
      width: 120,
      height: 120,
      doorId: "door3"
    };

    this.classes = [
      { class: GameEnvBackground, data: bgData },
      { class: TowerHUD, data: hudData },
      { class: Player, data: makePlayerData(path) },
      { class: PuzzleStation, data: stationData },
      { class: TowerDoor, data: doorData },
      { class: TreasureZone, data: treasureData }
    ];
  }
}

window.__towerCleanup = () => {
  const overlay = document.getElementById("tower-puzzle-overlay");
  if (overlay) overlay.remove();

  const fade = document.getElementById("tower-fade-overlay");
  if (fade) fade.remove();

  const styles = document.getElementById("tower-puzzle-styles");
  if (styles) styles.remove();
};

export const gameLevelClasses = [Floor1Level, Floor2Level, Floor3Level];
export { GameControl };